# Module 2 — Text Processing Lab

**Natural Language Processing**

## Nội dung
1. Xử lý chuỗi trong Python  
2. Unicode và encoding  
3. Đọc TXT và JSON  
4. Trích xuất văn bản từ PDF và DOCX  
5. HTML và trích xuất văn bản từ Web  
6. Regular Expressions  
7. Làm sạch văn bản phi cấu trúc  
8. Xây dựng Text Processing Pipeline  
9. Bài tập thực hành — **không có lời giải**

> Mục tiêu cuối buổi: xây dựng được pipeline  
> **Raw sources → Text extraction → Unicode normalization → Regex processing → Cleaning → Clean corpus**

### Quy định
- Sinh viên chạy lần lượt các cell ví dụ.
- Các phần có nhãn **BÀI TẬP** phải tự hoàn thiện.
- Notebook **không cung cấp lời giải** cho các bài tập.

## 0. Chuẩn bị môi trường

In [1]:
# Nếu chạy trên Google Colab, bỏ dấu # ở dòng dưới nếu cần cài thư viện.
# !pip install requests beautifulsoup4 pypdf python-docx

import re
import json
import unicodedata
from pathlib import Path
from collections import Counter

print("Environment is ready.")

Environment is ready.


# 1. Python Strings

Trong Python, văn bản thường được biểu diễn bởi kiểu `str`.

Các thao tác quan trọng trong NLP:
- indexing, slicing
- `lower()`, `upper()`
- `strip()`
- `split()`, `join()`
- `replace()`
- kiểm tra chuỗi với `in`, `startswith()`, `endswith()`

In [2]:
text = "  Natural Language Processing with Python  "

print(text)
print(text.strip())
print(text.lower())
print(text.upper())
print(text[2:9])

words = text.strip().split()
print(words)
print(" | ".join(words))

  Natural Language Processing with Python  
Natural Language Processing with Python
  natural language processing with python  
  NATURAL LANGUAGE PROCESSING WITH PYTHON  
Natural
['Natural', 'Language', 'Processing', 'with', 'Python']
Natural | Language | Processing | with | Python


### Chuỗi tiếng Việt

In [3]:
text_vi = "Xử lý ngôn ngữ tự nhiên là một lĩnh vực của Trí tuệ nhân tạo."

print("Số ký tự:", len(text_vi))
print("Chữ thường:", text_vi.lower())
print("Các từ:", text_vi.split())

Số ký tự: 61
Chữ thường: xử lý ngôn ngữ tự nhiên là một lĩnh vực của trí tuệ nhân tạo.
Các từ: ['Xử', 'lý', 'ngôn', 'ngữ', 'tự', 'nhiên', 'là', 'một', 'lĩnh', 'vực', 'của', 'Trí', 'tuệ', 'nhân', 'tạo.']


### BÀI TẬP 1 — String Processing

Cho chuỗi:

`"   Phenikaa University   |   Natural Language Processing   |  2026  "`

1. Loại bỏ khoảng trắng ở đầu/cuối.
2. Tách chuỗi thành 3 trường theo ký tự `|`.
3. Loại khoảng trắng thừa ở từng trường.
4. Chuyển tên học phần thành chữ thường.
5. Ghép ba trường lại bằng chuỗi `" - "`.
6. In số ký tự và số "từ" thu được bằng `split()`.

**Không sử dụng regular expression trong bài này.**

In [4]:
# TODO: Bài tập 1
text = "   Phenikaa University   |   Natural Language Processing   |  2026  "

# Viết code của bạn ở đây
print(text.strip())
print(text.split("|"))
word = [i.strip() for i in text.split("|")]
print(word)
print(word[1].lower())
word[1] = word[1].lower()
joined = " - ".join(word)

print(joined)
print("Số ký tự:", len(joined))  
print("Số từ:", sum(len(field.split()) for field in word)) 


Phenikaa University   |   Natural Language Processing   |  2026
['   Phenikaa University   ', '   Natural Language Processing   ', '  2026  ']
['Phenikaa University', 'Natural Language Processing', '2026']
natural language processing
Phenikaa University - natural language processing - 2026
Số ký tự: 56
Số từ: 6


# 2. Unicode and Encoding

Cần phân biệt:

**Character → Unicode code point → Encoding → Bytes**

Ví dụ một ký tự có thể được biểu diễn bằng code point Unicode, sau đó mã hóa thành UTF-8 để lưu dưới dạng bytes.

In [5]:
char = "ế"

print("Character:", char)
print("Code point:", hex(ord(char)))

encoded = char.encode("utf-8")
print("UTF-8 bytes:", encoded)

decoded = encoded.decode("utf-8")
print("Decoded:", decoded)

Character: ế
Code point: 0x1ebf
UTF-8 bytes: b'\xe1\xba\xbf'
Decoded: ế


## 2.1 `str` và `bytes`

In [6]:
text = "Tiếng Việt"

utf8 = text.encode("utf-8")

print(type(text), text)
print(type(utf8), utf8)
print(utf8.decode("utf-8"))

<class 'str'> Tiếng Việt
<class 'bytes'> b'Ti\xe1\xba\xbfng Vi\xe1\xbb\x87t'
Tiếng Việt


## 2.2 Unicode normalization

Hai chuỗi nhìn giống nhau có thể có biểu diễn Unicode khác nhau.

Các dạng normalization phổ biến:
- NFC
- NFD


In [7]:
s1 = "ế"
s2 = "e\u0302\u0301"

print(s1, [hex(ord(c)) for c in s1])
print(s2, [hex(ord(c)) for c in s2])
print("Equal before normalization:", s1 == s2)

n1 = unicodedata.normalize("NFC", s1)
n2 = unicodedata.normalize("NFC", s2)

print("Equal after NFC:", n1 == n2)

ế ['0x1ebf']
ế ['0x65', '0x302', '0x301']
Equal before normalization: False
Equal after NFC: True


### BÀI TẬP 2 — Unicode

1. Tạo hai chuỗi tiếng Việt nhìn giống nhau nhưng một chuỗi ở dạng NFC và một chuỗi ở dạng NFD.
2. So sánh:
   - `len()`
   - danh sách code point
   - kết quả phép `==`
3. Chuẩn hóa cả hai về NFC và so sánh lại.
4. Thử mã hóa một câu tiếng Việt bằng UTF-8 rồi decode trở lại.
5. Viết 2–3 câu nhận xét về lý do Unicode normalization quan trọng đối với NLP.

In [8]:
# TODO: Bài tập 2
text_nfc = "ế"
text_nfd = unicodedata.normalize("NFD", text_nfc)

print("NFC:", text_nfc)
print("NFD:", text_nfd)
print("len NFC:", len(text_nfc))
print("len NFD:", len(text_nfd))
print("Code points NFC:", [hex(ord(char)) for char in text_nfc])
print("Code points NFD:", [hex(ord(char)) for char in text_nfd])
print("Bằng nhau trước chuẩn hóa:", text_nfc == text_nfd)

nfc_from_nfc = unicodedata.normalize("NFC", text_nfc)
nfc_from_nfd = unicodedata.normalize("NFC", text_nfd)
print("Bằng nhau sau NFC:", nfc_from_nfc == nfc_from_nfd)

sentence = "Trí tuệ nhân tạo đang xử lý tiếng Việt."
encoded_sentence = sentence.encode("utf-8")
decoded_sentence = encoded_sentence.decode("utf-8")
print("Bytes UTF-8:", encoded_sentence)
print("Decode khôi phục đúng câu:", decoded_sentence == sentence)
print("Câu sau decode:", decoded_sentence)

NFC: ế
NFD: ế
len NFC: 1
len NFD: 3
Code points NFC: ['0x1ebf']
Code points NFD: ['0x65', '0x302', '0x301']
Bằng nhau trước chuẩn hóa: False
Bằng nhau sau NFC: True
Bytes UTF-8: b'Tr\xc3\xad tu\xe1\xbb\x87 nh\xc3\xa2n t\xe1\xba\xa1o \xc4\x91ang x\xe1\xbb\xad l\xc3\xbd ti\xe1\xba\xbfng Vi\xe1\xbb\x87t.'
Decode khôi phục đúng câu: True
Câu sau decode: Trí tuệ nhân tạo đang xử lý tiếng Việt.


# 3. Reading TXT and JSON

## 3.1 TXT

Luôn chỉ rõ encoding khi có thể, đặc biệt với dữ liệu tiếng Việt.

In [9]:
sample_path = Path("sample_vi.txt")

sample_text = '''Xử lý ngôn ngữ tự nhiên
Natural Language Processing
Dữ liệu văn bản có thể đến từ nhiều nguồn khác nhau.
'''

sample_path.write_text(sample_text, encoding="utf-8")

text = sample_path.read_text(encoding="utf-8")
print(text)

Xử lý ngôn ngữ tự nhiên
Natural Language Processing
Dữ liệu văn bản có thể đến từ nhiều nguồn khác nhau.



## 3.2 JSON

In [10]:
sample_json = {
    "id": "doc001",
    "title": "AI trong giáo dục",
    "author": "Nguyen Van A",
    "content": "Trí tuệ nhân tạo đang tạo ra nhiều thay đổi trong giáo dục.",
    "tags": ["AI", "education", "NLP"]
}

Path("sample.json").write_text(
    json.dumps(sample_json, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

data = json.loads(Path("sample.json").read_text(encoding="utf-8"))
print(data)
print("\nDocument text:")
print(data["title"] + "\n" + data["content"])

{'id': 'doc001', 'title': 'AI trong giáo dục', 'author': 'Nguyen Van A', 'content': 'Trí tuệ nhân tạo đang tạo ra nhiều thay đổi trong giáo dục.', 'tags': ['AI', 'education', 'NLP']}

Document text:
AI trong giáo dục
Trí tuệ nhân tạo đang tạo ra nhiều thay đổi trong giáo dục.


### BÀI TẬP 3 — TXT & JSON

Tạo một file JSON chứa ít nhất 5 bài viết. Mỗi bài gồm:
- `id`
- `title`
- `category`
- `content`

Yêu cầu:
1. Đọc JSON bằng Python.
2. Ghép `title` và `content` thành trường `text`.
3. Tính số ký tự của từng document.
4. Tính số từ thô bằng `split()`.
5. Tìm document dài nhất theo số ký tự.
6. Xuất kết quả thành một JSON mới.

Không dùng Pandas trong bài này.

In [11]:
# TODO: Bài tập 3
articles = [
    {
        "id": "post001",
        "title": "AI trong giáo dục",
        "category": "Công nghệ",
        "content": "Trí tuệ nhân tạo hỗ trợ cá nhân hóa bài học và giúp giáo viên theo dõi tiến độ học tập của sinh viên."
    },
    {
        "id": "post002",
        "title": "Bảo vệ môi trường",
        "category": "Môi trường",
        "content": "Mỗi người có thể giảm rác thải nhựa bằng cách dùng bình nước cá nhân, túi vải và phân loại rác tại nhà."
    },
    {
        "id": "post003",
        "title": "Thói quen đọc sách",
        "category": "Giáo dục",
        "content": "Đọc sách thường xuyên giúp mở rộng vốn từ, rèn luyện khả năng tập trung và tiếp cận nhiều góc nhìn mới."
    },
    {
        "id": "post004",
        "title": "An toàn trên Internet",
        "category": "Công nghệ",
        "content": "Người dùng nên đặt mật khẩu mạnh, bật xác thực hai yếu tố và cẩn trọng trước các liên kết đáng ngờ."
    },
    {
        "id": "post005",
        "title": "Lợi ích của vận động",
        "category": "Sức khỏe",
        "content": "Hoạt động thể chất đều đặn góp phần cải thiện sức khỏe tim mạch, giấc ngủ và tinh thần."
    }
]

input_path = Path("articles.json")
output_path = Path("articles_processed.json")

input_path.write_text(
    json.dumps(articles, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

# Đọc JSON, ghép title và content, rồi thống kê từng bài
loaded_articles = json.loads(input_path.read_text(encoding="utf-8"))
processed_articles = []

for article in loaded_articles:
    combined_text = article["title"] + " " + article["content"]
    processed_articles.append({
        **article,
        "text": combined_text,
        "num_chars": len(combined_text),
        "num_words_raw": len(combined_text.split())
    })

longest_article = max(processed_articles, key=lambda article: article["num_chars"])

# Xuất kết quả đã xử lý ra JSON mới
output_path.write_text(
    json.dumps(processed_articles, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

for article in processed_articles:
    print(
        article["id"],
        "| ký tự:", article["num_chars"],
        "| từ:", article["num_words_raw"]
    )

print("Bài dài nhất:", longest_article["id"], "-", longest_article["title"])

post001 | ký tự: 119 | từ: 28
post002 | ký tự: 121 | từ: 27
post003 | ký tự: 122 | từ: 26
post004 | ký tự: 121 | từ: 26
post005 | ký tự: 108 | từ: 24
Bài dài nhất: post003 - Thói quen đọc sách


# 4. Extracting Text from PDF and DOCX

Trong NLP, PDF/DOCX thường là **container/document format**. Ta cần trích xuất nội dung về dạng text trước khi xử lý.

> Lưu ý: PDF scan chỉ chứa ảnh sẽ cần OCR; phần OCR không phải trọng tâm của lab này.

## 4.1 PDF với `pypdf`

In [12]:
# Ví dụ sử dụng khi bạn có file PDF:
#
# from pypdf import PdfReader
#
# reader = PdfReader("document.pdf")
# pages = []
# for page in reader.pages:
#     pages.append(page.extract_text() or "")
#
# pdf_text = "\n".join(pages)
# print(pdf_text[:1000])

## 4.2 DOCX với `python-docx`

In [13]:
# Ví dụ sử dụng khi bạn có file DOCX:
#
# from docx import Document
#
# doc = Document("document.docx")
# paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
# docx_text = "\n".join(paragraphs)
# print(docx_text[:1000])

### BÀI TẬP 4 — Document Extraction

Chuẩn bị:
- 01 file PDF có text
- 01 file DOCX

Yêu cầu:
1. Viết `extract_pdf(path)`.
2. Viết `extract_docx(path)`.
3. Mỗi hàm trả về một chuỗi duy nhất.
4. Thống kê cho từng tài liệu:
   - số trang/đoạn nếu lấy được;
   - số ký tự;
   - số dòng;
   - 20 từ xuất hiện nhiều nhất theo cách tách bằng whitespace.
5. Nhận xét ít nhất 2 vấn đề gặp phải khi trích xuất text từ document.


In [14]:
# TODO: Bài tập 4
from collections import Counter
from pypdf import PdfReader
from docx import Document

# 1. Hàm trích xuất PDF
def extract_pdf(path):
    reader = PdfReader(path)
    return "\n".join([page.extract_text() or "" for page in reader.pages])

# 2. Hàm trích xuất DOCX
def extract_docx(path):
    doc = Document(path)
    return "\n".join([p.text for p in doc.paragraphs if p.text.strip()])

# 3. Hàm thống kê tài liệu ngắn gọn
def thong_ke(ten_file, text, so_trang_doan=None):
    print(f"=== Thống kê: {ten_file} ===")
    if so_trang_doan:
        print(f"- Số trang/đoạn: {so_trang_doan}")
    print(f"- Số ký tự: {len(text)}")
    print(f"- Số dòng: {len([line for line in text.splitlines() if line.strip()])}")
    words = text.split()
    print(f"- Top 20 từ xuất hiện nhiều nhất: {Counter(words).most_common(20)}\n")

# Thực hiện trích xuất và in thống kê
pdf_text = extract_pdf("sample.pdf")
thong_ke("sample.pdf", pdf_text, len(PdfReader("sample.pdf").pages))

docx_text = extract_docx("sample.docx")
thong_ke("sample.docx", docx_text, len([p for p in Document("sample.docx").paragraphs if p.text.strip()]))

# 4. Nhận xét 2 vấn đề khi trích xuất text:
print("""
Nhận xét 2 vấn đề khi trích xuất text từ document:
1. Mất thứ tự đọc và bố cục (Layout): File PDF có nhiều cột hoặc bảng biểu dễ bị xáo trộn thứ tự dòng và dính text từ header/footer.
2. Bỏ sót nội dung: Với DOCX, đọc qua paragraph cơ bản sẽ bỏ qua text trong bảng (table) và text box; với PDF scan thì không trích xuất được nếu thiếu OCR.
""".strip())


incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


=== Thống kê: sample.pdf ===
- Số trang/đoạn: 2
- Số ký tự: 1046
- Số dòng: 18
- Top 20 từ xuất hiện nhiều nhất: [('and', 8), ('Text', 4), ('text', 4), ('of', 3), ('Natural', 2), ('Language', 2), ('Processing', 2), ('2:', 2), ('is', 2), ('in', 2), ('1.', 2), ('from', 2), ('extraction', 2), ('can', 2), ('End', 2), ('Page', 2), ('model', 2), ('(NLP)', 1), ('Lab', 1), ('Module', 1)]

=== Thống kê: sample.docx ===
- Số trang/đoạn: 7
- Số ký tự: 962
- Số dòng: 7
- Top 20 từ xuất hiện nhiều nhất: [('lý', 6), ('và', 6), ('ngữ', 5), ('bản', 5), ('văn', 5), ('ngôn', 3), ('là', 3), ('của', 3), ('bước', 3), ('trong', 3), ('xử', 3), ('các', 3), ('mô', 3), ('hình', 3), ('Xử', 2), ('nhiên', 2), ('tự', 2), ('Processing', 2), ('một', 2), ('tạo', 2)]

Nhận xét 2 vấn đề khi trích xuất text từ document:
1. Mất thứ tự đọc và bố cục (Layout): File PDF có nhiều cột hoặc bảng biểu dễ bị xáo trộn thứ tự dòng và dính text từ header/footer.
2. Bỏ sót nội dung: Với DOCX, đọc qua paragraph cơ bản sẽ bỏ qua text t

# 5. HTML and Web Text Extraction

Một pipeline đơn giản:

**URL → HTTP response → HTML → DOM parsing → relevant elements → text**

In [15]:
from bs4 import BeautifulSoup

html = '''
<html>
  <head>
    <title>NLP Demo</title>
    <style>.hidden {display:none}</style>
  </head>
  <body>
    <nav>Home | About | Contact</nav>
    <article>
      <h1>Natural Language Processing</h1>
      <p>NLP helps computers process human language.</p>
      <p>Text processing is an important first step.</p>
    </article>
    <script>console.log("not article text")</script>
  </body>
</html>
'''

soup = BeautifulSoup(html, "html.parser")

print("Title:", soup.title.get_text(strip=True))
article = soup.find("article")
print(article.get_text(" ", strip=True))

Title: NLP Demo
Natural Language Processing NLP helps computers process human language. Text processing is an important first step.


## 5.1 Request một trang web

In [16]:
# Chỉ chạy với website cho phép truy cập và khi môi trường có Internet.
#
# import requests
#
# url = "https://example.com"
# response = requests.get(
#     url,
#     timeout=10,
#     headers={"User-Agent": "NLP-course-demo/1.0"}
# )
# response.raise_for_status()
#
# soup = BeautifulSoup(response.text, "html.parser")
# print(soup.get_text(" ", strip=True)[:1000])

### BÀI TẬP 5 — Web Text Extraction

Chọn **3 bài viết** từ một website tin tức hoặc website công khai phù hợp với mục đích học tập.

Yêu cầu:
1. Lấy HTML của từng URL.
2. Trích xuất:
   - title
   - các đoạn văn chính
   - URL
3. Không lấy nội dung từ `script` và `style`.
4. Lưu thành JSON.
5. So sánh text lấy bằng `soup.get_text()` với text chỉ lấy trong vùng bài viết.
6. Giải thích tại sao "lấy toàn bộ text trong HTML" thường tạo dữ liệu nhiễu.

Tuân thủ điều khoản sử dụng, `robots.txt` và giới hạn truy cập của website.

In [17]:
# TODO: Bài tập 5
import json
from pathlib import Path
import requests
from bs4 import BeautifulSoup

urls = [
    "https://vi.wikipedia.org/wiki/X%E1%BB%AD_l%C3%BD_ng%C3%B4n_ng%E1%BB%AF_t%E1%BB%B1_nhi%C3%AAn",
    "https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o",
    "https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_m%C3%A1y"
]

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) NLP-Lab-Student/1.0 (Educational)"
}

extracted_articles = []
comparison_results = []

for idx, url in enumerate(urls, start=1):
    print(f"--> Đang cào dữ liệu từ URL {idx}: {url}")
    response = requests.get(url, headers=headers, timeout=10)
    response.raise_for_status()
    html_content = response.text
    
    soup = BeautifulSoup(html_content, "html.parser")

    raw_full_text = soup.get_text(" ", strip=True)
    
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
        
    title_tag = soup.find("h1", id="firstHeading") or soup.find("h1")
    title = title_tag.get_text(strip=True) if title_tag else (soup.title.get_text(strip=True) if soup.title else "No Title")
    
    content_area = soup.find("div", id="mw-content-text") or soup.find("article") or soup.body
    
    paragraphs = []
    if content_area:
        for p in content_area.find_all("p"):
            p_text = p.get_text(" ", strip=True)
            if len(p_text) > 30: 
                paragraphs.append(p_text)
                
    article_text = "\n\n".join(paragraphs)
    
    doc_data = {
        "id": f"web_doc_{idx:03d}",
        "url": url,
        "title": title,
        "paragraphs": paragraphs,
        "full_article_text": article_text,
        "num_paragraphs": len(paragraphs),
        "num_chars": len(article_text)
    }
    extracted_articles.append(doc_data)
    
    comparison_results.append({
        "title": title,
        "raw_full_chars": len(raw_full_text),
        "article_only_chars": len(article_text),
        "raw_full_words": len(raw_full_text.split()),
        "article_only_words": len(article_text.split())
    })


output_json_path = Path("web_articles.json")
output_json_path.write_text(
    json.dumps(extracted_articles, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print(f"\n[OK] Đã lưu {len(extracted_articles)} bài viết vào '{output_json_path}'.\n")

print("=" * 70)
print("BẢNG SO SÁNH: soup.get_text() TOÀN BỘ TRANG vs TEXT CHỈ TRONG VÙNG BÀI VIẾT")
print("=" * 70)
for res in comparison_results:
    print(f"Tiêu đề: {res['title']}")
    print(f"  - Toàn trang (soup.get_text): {res['raw_full_chars']} ký tự | {res['raw_full_words']} từ")
    print(f"  - Vùng bài viết chính:        {res['article_only_chars']} ký tự | {res['article_only_words']} từ")
    diff_chars = res['raw_full_chars'] - res['article_only_chars']
    print(f"  --> Chênh lệch nhiễu (Noise): {diff_chars} ký tự (~{diff_chars / res['raw_full_chars'] * 100:.1f}% toàn trang)\n")




--> Đang cào dữ liệu từ URL 1: https://vi.wikipedia.org/wiki/X%E1%BB%AD_l%C3%BD_ng%C3%B4n_ng%E1%BB%AF_t%E1%BB%B1_nhi%C3%AAn
--> Đang cào dữ liệu từ URL 2: https://vi.wikipedia.org/wiki/Tr%C3%AD_tu%E1%BB%87_nh%C3%A2n_t%E1%BA%A1o
--> Đang cào dữ liệu từ URL 3: https://vi.wikipedia.org/wiki/H%E1%BB%8Dc_m%C3%A1y

[OK] Đã lưu 3 bài viết vào 'web_articles.json'.

BẢNG SO SÁNH: soup.get_text() TOÀN BỘ TRANG vs TEXT CHỈ TRONG VÙNG BÀI VIẾT
Tiêu đề: Xử lý ngôn ngữ tự nhiên
  - Toàn trang (soup.get_text): 19987 ký tự | 4305 từ
  - Vùng bài viết chính:        651 ký tự | 144 từ
  --> Chênh lệch nhiễu (Noise): 19336 ký tự (~96.7% toàn trang)

Tiêu đề: Trí tuệ nhân tạo
  - Toàn trang (soup.get_text): 86896 ký tự | 17774 từ
  - Vùng bài viết chính:        21967 ký tự | 4986 từ
  --> Chênh lệch nhiễu (Noise): 64929 ký tự (~74.7% toàn trang)

Tiêu đề: Học máy
  - Toàn trang (soup.get_text): 50117 ký tự | 10226 từ
  - Vùng bài viết chính:        17927 ký tự | 4023 từ
  --> Chênh lệch nhiễu (Noise): 321

# 6. Regular Expressions

Regex được dùng để mô tả và tìm kiếm **patterns** trong text.

Một số thành phần cơ bản:
- `.`: một ký tự
- `\\d`: chữ số
- `\\w`: ký tự word
- `\\s`: whitespace
- `+`, `*`, `?`, `{m,n}`: quantifiers
- `^`, `$`: anchors
- `(...)`: group
- `|`: alternation

In [18]:
text = '''
Liên hệ: student@phenikaa-uni.edu.vn
Hotline: 024-1234-5678
Website: https://phenikaa-uni.edu.vn
Ngày cập nhật: 26/09/2026
'''

emails = re.findall(r'[\w.-]+@[\w.-]+\.\w+', text)
urls = re.findall(r'https?://\S+', text)
dates = re.findall(r'\b\d{2}/\d{2}/\d{4}\b', text)

print("Emails:", emails)
print("URLs:", urls)
print("Dates:", dates)

Emails: ['student@phenikaa-uni.edu.vn']
URLs: ['https://phenikaa-uni.edu.vn']
Dates: ['26/09/2026']


## 6.1 `search`, `findall`, `sub`

In [19]:
text = "NLP   is\tinteresting.\nText processing   matters."

print(re.search(r"NLP", text))
print(re.findall(r"\b\w+ing\b", text))
print(re.sub(r"\s+", " ", text).strip())

<re.Match object; span=(0, 3), match='NLP'>
['interesting', 'processing']
NLP is interesting. Text processing matters.


### BÀI TẬP 6 — Regex

Cho một tập văn bản chứa thông tin liên hệ và nội dung bài viết.

Viết regex để:
1. Trích xuất email.
2. Trích xuất URL bắt đầu bằng `http://` hoặc `https://`.
3. Trích xuất ngày theo `dd/mm/yyyy`.
4. Trích xuất số điện thoại theo ít nhất 2 format do bạn tự định nghĩa.
5. Tìm các số nguyên và số thực.
6. Thay mọi chuỗi whitespace liên tiếp bằng một dấu cách.
7. Thay email bằng token `<EMAIL>`.
8. Thay URL bằng token `<URL>`.

Với mỗi regex, hãy bổ sung ít nhất **2 test cases**.

In [20]:
# TODO: Bài tập 6
import re

sample = '''
Họ tên: Nguyen Van A
Email: nva@example.com
Ngày: 26/09/2026
Website: https://example.com/article?id=123
Điện thoại: 0912 345 678
Điểm: 8.75
'''

# ------------------------------------------------------------------------------
# 1. Trích xuất email
# ------------------------------------------------------------------------------
email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
emails_sample = re.findall(email_pattern, sample)
print("1. Email từ sample:", emails_sample)
# 2 test cases bổ sung:
print("   - Test case 1.1:", re.findall(email_pattern, "Gửi bài về: sinhvien.nlp@phenikaa-uni.edu.vn."))
print("   - Test case 1.2:", re.findall(email_pattern, "Liên hệ contact@sub.domain.org hoặc admin@ai.vn"))
print()

# ------------------------------------------------------------------------------
# 2. Trích xuất URL bắt đầu bằng http:// hoặc https://
# ------------------------------------------------------------------------------
url_pattern = r'https?://[^\s,;<>"\')]+'
urls_sample = re.findall(url_pattern, sample)
print("2. URL từ sample:", urls_sample)
# 2 test cases bổ sung:
print("   - Test case 2.1:", re.findall(url_pattern, "Tài liệu tại: http://nlp.phenikaa-uni.edu.vn/lab-2"))
print("   - Test case 2.2:", re.findall(url_pattern, "Xem mô hình: https://huggingface.co/models?filter=text"))
print()

# ------------------------------------------------------------------------------
# 3. Trích xuất ngày theo dd/mm/yyyy
# ------------------------------------------------------------------------------
date_pattern = r'\b(0[1-9]|[12][0-9]|3[01])/(0[1-9]|1[0-2])/\d{4}\b'
dates_sample = [m.group(0) for m in re.finditer(date_pattern, sample)]
print("3. Ngày (dd/mm/yyyy) từ sample:", dates_sample)
# 2 test cases bổ sung:
print("   - Test case 3.1:", [m.group(0) for m in re.finditer(date_pattern, "Khai giảng ngày 05/10/2026 và kết thúc 15/12/2026.")])
print("   - Test case 3.2:", [m.group(0) for m in re.finditer(date_pattern, "Hạn cuối nộp báo cáo: 01/01/2027.")])
print()

# ------------------------------------------------------------------------------
# 4. Trích xuất số điện thoại (2 formats: di động VN 09xx và có mã +84 / máy bàn)
# ------------------------------------------------------------------------------
phone_pattern = r'(?:\+84|0)(?:[\s.-]?\d{2,3})(?:[\s.-]?\d{3})(?:[\s.-]?\d{3,4})|\(\d{2,4}\)[\s.-]?\d{3,4}[\s.-]?\d{3,4}'
phones_sample = re.findall(phone_pattern, sample)
print("4. Số điện thoại từ sample:", phones_sample)
# 2 test cases bổ sung:
print("   - Test case 4.1 (Format có dấu gạch/chấm):", re.findall(phone_pattern, "Hotline: 0988-123-456 hoặc 0901.234.567."))
print("   - Test case 4.2 (Format +84 và cố định):", re.findall(phone_pattern, "Gọi +84 912 345 678 hoặc máy bàn (024) 3754 7890."))
print()

# ------------------------------------------------------------------------------
# 5. Tìm các số nguyên và số thực
# ------------------------------------------------------------------------------
number_pattern = r'-?\b\d+\.\d+\b|-?\b\d+\b'
numbers_sample = re.findall(number_pattern, sample)
print("5. Số nguyên và số thực từ sample:", numbers_sample)
# 2 test cases bổ sung:
print("   - Test case 5.1:", re.findall(number_pattern, "Điểm trung bình là 9.25 trên tổng số 50 câu hỏi."))
print("   - Test case 5.2:", re.findall(number_pattern, "Nhiệt độ dao động từ -3.5 độ C đến 40 độ C với bước nhảy 1."))
print()

# ------------------------------------------------------------------------------
# 6. Thay mọi chuỗi whitespace liên tiếp bằng một dấu cách
# ------------------------------------------------------------------------------
clean_ws_sample = re.sub(r'\s+', ' ', sample).strip()
print("6. Chuẩn hóa whitespace của sample:\n  ", clean_ws_sample)
# 2 test cases bổ sung:
print("   - Test case 6.1:", re.sub(r'\s+', ' ', "NLP   is \t\t great!\n\nText processing matters.").strip())
print("   - Test case 6.2:", re.sub(r'\s+', ' ', "   Khoa học   dữ liệu   và   AI.   ").strip())
print()

# ------------------------------------------------------------------------------
# 7. Thay email bằng token <EMAIL>
# ------------------------------------------------------------------------------
masked_email_sample = re.sub(email_pattern, '<EMAIL>', sample)
print("7. Thay email trong sample thành <EMAIL>:\n" + masked_email_sample.strip())
# 2 test cases bổ sung:
print("   - Test case 7.1:", re.sub(email_pattern, "<EMAIL>", "Mail cá nhân: user@gmail.com, mail cty: hr@company.vn"))
print("   - Test case 7.2:", re.sub(email_pattern, "<EMAIL>", "Liên hệ admin: support.team@phenikaa.edu.vn."))
print()

# ------------------------------------------------------------------------------
# 8. Thay URL bằng token <URL>
# ------------------------------------------------------------------------------
masked_url_sample = re.sub(url_pattern, '<URL>', sample)
print("8. Thay URL trong sample thành <URL>:\n" + masked_url_sample.strip())
# 2 test cases bổ sung:
print("   - Test case 8.1:", re.sub(url_pattern, "<URL>", "Truy cập https://google.com hoặc http://github.com/nlp"))
print("   - Test case 8.2:", re.sub(url_pattern, "<URL>", "Chi tiết tại https://example.com/item?id=99 (tham khảo)."))


1. Email từ sample: ['nva@example.com']
   - Test case 1.1: ['sinhvien.nlp@phenikaa-uni.edu.vn']
   - Test case 1.2: ['contact@sub.domain.org', 'admin@ai.vn']

2. URL từ sample: ['https://example.com/article?id=123']
   - Test case 2.1: ['http://nlp.phenikaa-uni.edu.vn/lab-2']
   - Test case 2.2: ['https://huggingface.co/models?filter=text']

3. Ngày (dd/mm/yyyy) từ sample: ['26/09/2026']
   - Test case 3.1: ['05/10/2026', '15/12/2026']
   - Test case 3.2: ['01/01/2027']

4. Số điện thoại từ sample: ['0912 345 678']
   - Test case 4.1 (Format có dấu gạch/chấm): ['0988-123-456', '0901.234.567']
   - Test case 4.2 (Format +84 và cố định): ['+84 912 345 678', '(024) 3754 7890']

5. Số nguyên và số thực từ sample: ['26', '09', '2026', '123', '0912', '345', '678', '8.75']
   - Test case 5.1: ['9.25', '50']
   - Test case 5.2: ['-3.5', '40', '1']

6. Chuẩn hóa whitespace của sample:
   Họ tên: Nguyen Van A Email: nva@example.com Ngày: 26/09/2026 Website: https://example.com/article?id=123 Đi

# 7. Cleaning Unstructured Text

Không có một quy trình cleaning đúng cho mọi bài toán.

Ví dụ:
- sentiment analysis có thể cần emoji, dấu `!`;
- NER có thể cần chữ hoa/chữ thường;
- topic modeling truyền thống thường cần chuẩn hóa mạnh hơn.

Vì vậy cần phân biệt:
**noise** và **information useful for the downstream task**.

In [21]:
raw_text = '''
   <p>Phenikaa University</p>

   NLP is GREAT!!! 😀😀
   Contact: nlp@example.com
   More info: https://example.com/nlp

'''

def basic_clean(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+", " <URL> ", text)
    text = re.sub(r"[\w.-]+@[\w.-]+\.\w+", " <EMAIL> ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

print("RAW:")
print(raw_text)

print("\nCLEAN:")
print(basic_clean(raw_text))

RAW:

   <p>Phenikaa University</p>

   NLP is GREAT!!! 😀😀
   Contact: nlp@example.com
   More info: https://example.com/nlp



CLEAN:
Phenikaa University NLP is GREAT!!! 😀😀 Contact: <EMAIL> More info: <URL>


### BÀI TẬP 7 — Thiết kế `clean_text()`

Viết hàm:

`clean_text(text, lowercase=False, remove_urls=False, remove_emails=False)`

Yêu cầu tối thiểu:
1. Unicode normalize về NFC.
2. Xử lý HTML tags.
3. Chuẩn hóa whitespace.
4. Cho phép tùy chọn lowercase.
5. Cho phép tùy chọn xóa/thay URL.
6. Cho phép tùy chọn xóa/thay email.
7. Không làm mất dấu tiếng Việt.
8. Tạo ít nhất 5 test cases gồm:
   - tiếng Việt;
   - HTML;
   - URL/email;
   - nhiều whitespace;
   - emoji/ký tự đặc biệt.

**Quan trọng:** giải thích ngắn gọn những thông tin nào bạn quyết định giữ lại và tại sao.

In [22]:
# TODO: Bài tập 7
import unicodedata
import re
import html

def clean_text(text, lowercase=False, remove_urls=False, remove_emails=False, replace_tokens=False):
    """
    Hàm tiền xử lý và làm sạch văn bản cho NLP pipeline.
    
    Tham số:
    - text (str): Chuỗi văn bản đầu vào.
    - lowercase (bool): Chuyển văn bản về chữ thường nếu True.
    - remove_urls (bool): Xóa URL (hoặc thay bằng <URL>) nếu True.
    - remove_emails (bool): Xóa Email (hoặc thay bằng <EMAIL>) nếu True.
    - replace_tokens (bool): Nếu True, thay URL/Email bằng token thay vì xóa hẳn.
    
    Trả về:
    - str: Chuỗi văn bản đã được làm sạch và chuẩn hóa.
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Unicode Normalization về dạng NFC (chuẩn dựng sẵn cho tiếng Việt)
    text = unicodedata.normalize("NFC", text)
    
    # 2. Giải mã HTML entities (ví dụ: &amp; -> &, &lt; -> <) và xóa HTML tags
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3. Xử lý URLs
    if remove_urls:
        replacement = " <URL> " if replace_tokens else " "
        text = re.sub(r'https?://[^\s,;<>"\')]+', replacement, text)
        
    # 4. Xử lý Emails
    if remove_emails:
        replacement = " <EMAIL> " if replace_tokens else " "
        text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', replacement, text)
        
    # 5. Tùy chọn chuyển chữ thường (Lowercase)
    if lowercase:
        text = text.lower()
        
    # 6. Chuẩn hóa khoảng trắng (Whitespace normalization)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# ==============================================================================
# 5 TEST CASES KIỂM THỬ
# ==============================================================================

# Test Case 1: Tiếng Việt có dấu (Bảo toàn nguyên vẹn dấu tiếng Việt)
tc1_vietnamese = "Trường Đại học Phenikaa đào tạo ngành Xử lý Ngôn ngữ Tự nhiên và Trí tuệ Nhân tạo."
print("--- TEST CASE 1: Tiếng Việt ---")
print("Input :", tc1_vietnamese)
print("Output:", clean_text(tc1_vietnamese, lowercase=True))
print()

# Test Case 2: HTML tags lồng nhau & HTML entities
tc2_html = "<div class='content'><h1>Khóa học <i>NLP</i></h1><p>Nội dung &amp; bài tập thực hành lab.</p></div>"
print("--- TEST CASE 2: HTML Tags & Entities ---")
print("Input :", tc2_html)
print("Output:", clean_text(tc2_html))
print()

# Test Case 3: Chứa URL và Email (Kiểm tra cờ remove_urls và remove_emails)
tc3_contact = "Liên hệ giảng viên qua email nva@phenikaa-uni.edu.vn hoặc website https://nlp.example.com/course."
print("--- TEST CASE 3: URL & Email ---")
print("Input       :", tc3_contact)
print("Output (Xóa):", clean_text(tc3_contact, remove_urls=True, remove_emails=True))
print("Output (Thay token):", clean_text(tc3_contact, remove_urls=True, remove_emails=True, replace_tokens=True))
print()

# Test Case 4: Chuỗi nhiều whitespace (dấu cách liên tiếp, newline, tabs)
tc4_whitespace = "   Xử lý     văn bản \t\t với   nhiều \n\n khoảng trắng   thừa.   "
print("--- TEST CASE 4: Whitespace phức tạp ---")
print("Input :", repr(tc4_whitespace))
print("Output:", repr(clean_text(tc4_whitespace)))
print()






--- TEST CASE 1: Tiếng Việt ---
Input : Trường Đại học Phenikaa đào tạo ngành Xử lý Ngôn ngữ Tự nhiên và Trí tuệ Nhân tạo.
Output: trường đại học phenikaa đào tạo ngành xử lý ngôn ngữ tự nhiên và trí tuệ nhân tạo.

--- TEST CASE 2: HTML Tags & Entities ---
Input : <div class='content'><h1>Khóa học <i>NLP</i></h1><p>Nội dung &amp; bài tập thực hành lab.</p></div>
Output: Khóa học NLP Nội dung & bài tập thực hành lab.

--- TEST CASE 3: URL & Email ---
Input       : Liên hệ giảng viên qua email nva@phenikaa-uni.edu.vn hoặc website https://nlp.example.com/course.
Output (Xóa): Liên hệ giảng viên qua email hoặc website
Output (Thay token): Liên hệ giảng viên qua email <EMAIL> hoặc website <URL>

--- TEST CASE 4: Whitespace phức tạp ---
Input : '   Xử lý     văn bản \t\t với   nhiều \n\n khoảng trắng   thừa.   '
Output: 'Xử lý văn bản với nhiều khoảng trắng thừa.'



# 8. Building a Text Processing Pipeline

Mục tiêu là chuẩn hóa nhiều nguồn dữ liệu về một schema chung:

```text
TXT ─────┐
PDF ─────┤
DOCX ────┤
JSON ────┼──> extract_text()
HTML ────┘
                  ↓
          normalize_unicode()
                  ↓
             clean_text()
                  ↓
             CLEAN CORPUS
```

In [23]:
def normalize_unicode(text, form="NFC"):
    return unicodedata.normalize(form, text)

def make_document(doc_id, source, raw_text, title="", url=""):
    return {
        "id": doc_id,
        "source": source,
        "title": title,
        "url": url,
        "raw_text": raw_text,
        "clean_text": basic_clean(raw_text),
        "num_chars": len(raw_text),
        "num_words_raw": len(raw_text.split())
    }

doc = make_document(
    "demo01",
    "txt",
    "  Xử lý   ngôn ngữ tự nhiên.  ",
    title="Demo"
)

print(json.dumps(doc, ensure_ascii=False, indent=2))

{
  "id": "demo01",
  "source": "txt",
  "title": "Demo",
  "url": "",
  "raw_text": "  Xử lý   ngôn ngữ tự nhiên.  ",
  "clean_text": "Xử lý ngôn ngữ tự nhiên.",
  "num_chars": 30,
  "num_words_raw": 6
}


# 9. BÀI TẬP TỔNG HỢP — Mini Project

## Building a Text Acquisition & Cleaning Pipeline

Thu thập hoặc chuẩn bị **20–50 documents** từ ít nhất **4 loại nguồn** trong số:

- TXT
- JSON
- PDF
- DOCX
- HTML/Web

### Yêu cầu A — Extraction
Xây dựng các hàm phù hợp, ví dụ:

```python
extract_txt(path)
extract_json(path)
extract_pdf(path)
extract_docx(path)
extract_html(html)
```

Mọi nguồn cuối cùng phải đưa về text.

### Yêu cầu B — Common document schema

Mỗi document phải có ít nhất:

```python
{
    "id": "...",
    "source": "...",
    "title": "...",
    "url": "...",
    "raw_text": "...",
    "clean_text": "...",
    "num_chars": ...,
    "num_words_raw": ...
}
```

### Yêu cầu C — Cleaning pipeline

Pipeline phải có:
1. Unicode normalization.
2. HTML handling.
3. URL/email handling.
4. Whitespace normalization.
5. Ít nhất 2 quy tắc regex do nhóm tự thiết kế.
6. Cấu hình lowercase bật/tắt được.

### Yêu cầu D — Corpus statistics

Báo cáo:
- số documents theo source;
- tổng số ký tự;
- trung bình số ký tự/document;
- document dài nhất/ngắn nhất;
- 20 token thô xuất hiện nhiều nhất;
- số URL và email tìm được trước cleaning.

### Yêu cầu E — Quality inspection

Chọn ngẫu nhiên ít nhất 5 documents và trình bày:
- raw text;
- clean text;
- những thay đổi quan trọng;
- trường hợp cleaning có thể làm mất thông tin.

### Yêu cầu F — Output

Lưu corpus sau xử lý thành:

`clean_corpus.json`

Không cần thực hiện stemming, lemmatization hay xây dựng mô hình ML trong Module 2.

In [24]:
# ==============================================================================
# BÀI TẬP TỔNG HỢP — MINI PROJECT: Text Acquisition & Cleaning Pipeline
# ==============================================================================
import os
import json
import re
import random
import unicodedata
from pathlib import Path
from collections import Counter
from pypdf import PdfReader
from docx import Document

# ------------------------------------------------------------------------------
# YÊU CẦU A — CÁC HÀM EXTRACTION (Trích xuất văn bản từ nhiều nguồn)
# ------------------------------------------------------------------------------
def extract_txt(path):
    """Đọc text từ file TXT"""
    return Path(path).read_text(encoding="utf-8")

def extract_json(path):
    """Đọc danh sách bài viết từ file JSON"""
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    return data  # Trả về list các dict

def extract_pdf(path):
    """Trích xuất text từ file PDF"""
    reader = PdfReader(path)
    pages = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(pages)

def extract_docx(path):
    """Trích xuất text từ file DOCX"""
    doc = Document(path)
    paragraphs = [p.text for p in doc.paragraphs if p.text.strip()]
    return "\n".join(paragraphs)

def extract_html(html_str):
    """Lấy text từ chuỗi HTML đơn giản bằng regex"""
    text = re.sub(r'<script.*?</script>', ' ', html_str, flags=re.DOTALL)
    text = re.sub(r'<style.*?</style>', ' ', text, flags=re.DOTALL)
    text = re.sub(r'<[^>]+>', ' ', text)
    return text

# ------------------------------------------------------------------------------
# YÊU CẦU C — CLEANING PIPELINE
# ------------------------------------------------------------------------------
def clean_text_pipeline(text, lowercase=False):
    """
    Pipeline làm sạch văn bản:
    1. Chuẩn hóa Unicode NFC
    2. Xử lý HTML tags
    3. Thay URL và Email bằng token
    4. Regex 1: Xóa các dấu câu lặp lại vô nghĩa (vd: !!!, ???, ...)
    5. Regex 2: Loại bỏ ký tự icon lạ/rác ngoài bảng chữ số và dấu câu thông thường
    6. Tùy chọn Lowercase
    7. Chuẩn hóa khoảng trắng Whitespace
    """
    # 1. Unicode Normalization
    text = unicodedata.normalize("NFC", text)
    
    # 2. Xử lý HTML
    text = re.sub(r'<[^>]+>', ' ', text)
    
    # 3. URL & Email handling
    text = re.sub(r'https?://[^\s,;<>"\')]+', ' <URL> ', text)
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', ' <EMAIL> ', text)
    
    # 4. Regex tự thiết kế 1: Rút gọn các dấu câu bị lặp liên tiếp (ví dụ "!!!" -> "!", "???" -> "?")
    text = re.sub(r'([!?.])\1+', r'\1', text)
    
    # 5. Regex tự thiết kế 2: Xóa các ký tự đặc biệt rác/không mong muốn (giữ lại chữ, số, dấu câu cơ bản)
    text = re.sub(r'[^\w\s.,!?:;()/\-–<>]', '', text)
    
    # 6. Lowercase (nếu bật)
    if lowercase:
        text = text.lower()
        
    # 7. Whitespace normalization
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ------------------------------------------------------------------------------
# YÊU CẦU B — THU THẬP VÀ CHUẨN HÓA VỀ COMMON DOCUMENT SCHEMA
# ------------------------------------------------------------------------------
corpus = []
doc_counter = 1

# 1. Thu thập từ TXT (thư mục data/txt)
for p in sorted(Path("data/txt").glob("*.txt")):
    raw = extract_txt(p)
    clean = clean_text_pipeline(raw, lowercase=True)
    corpus.append({
        "id": f"doc_{doc_counter:03d}",
        "source": "TXT",
        "title": p.stem,
        "url": "",
        "raw_text": raw,
        "clean_text": clean,
        "num_chars": len(raw),
        "num_words_raw": len(raw.split())
    })
    doc_counter += 1

# 2. Thu thập từ JSON (file data/corpus_articles.json)
if Path("data/corpus_articles.json").exists():
    articles = extract_json("data/corpus_articles.json")
    for item in articles:
        raw = item.get("title", "") + ". " + item.get("content", "")
        clean = clean_text_pipeline(raw, lowercase=True)
        corpus.append({
            "id": f"doc_{doc_counter:03d}",
            "source": "JSON",
            "title": item.get("title", ""),
            "url": "",
            "raw_text": raw,
            "clean_text": clean,
            "num_chars": len(raw),
            "num_words_raw": len(raw.split())
        })
        doc_counter += 1

# 3. Thu thập từ PDF (thư mục data/pdf)
for p in sorted(Path("data/pdf").glob("*.pdf")):
    raw = extract_pdf(p)
    clean = clean_text_pipeline(raw, lowercase=True)
    corpus.append({
        "id": f"doc_{doc_counter:03d}",
        "source": "PDF",
        "title": p.stem,
        "url": "",
        "raw_text": raw,
        "clean_text": clean,
        "num_chars": len(raw),
        "num_words_raw": len(raw.split())
    })
    doc_counter += 1

# 4. Thu thập từ DOCX (thư mục data/docx)
for p in sorted(Path("data/docx").glob("*.docx")):
    raw = extract_docx(p)
    clean = clean_text_pipeline(raw, lowercase=True)
    corpus.append({
        "id": f"doc_{doc_counter:03d}",
        "source": "DOCX",
        "title": p.stem,
        "url": "",
        "raw_text": raw,
        "clean_text": clean,
        "num_chars": len(raw),
        "num_words_raw": len(raw.split())
    })
    doc_counter += 1

# 5. Thu thập từ Web/HTML (file web_articles.json nếu có)
if Path("web_articles.json").exists():
    web_data = extract_json("web_articles.json")
    for item in web_data:
        raw = item.get("title", "") + ". " + item.get("full_article_text", "")
        clean = clean_text_pipeline(raw, lowercase=True)
        corpus.append({
            "id": f"doc_{doc_counter:03d}",
            "source": "HTML/Web",
            "title": item.get("title", ""),
            "url": item.get("url", ""),
            "raw_text": raw,
            "clean_text": clean,
            "num_chars": len(raw),
            "num_words_raw": len(raw.split())
        })
        doc_counter += 1

print(f"--> Tổng số documents thu thập được: {len(corpus)}")

# ------------------------------------------------------------------------------
# YÊU CẦU D — CORPUS STATISTICS (Thống kê bộ dữ liệu)
# ------------------------------------------------------------------------------
sources_count = Counter([doc["source"] for doc in corpus])
total_chars = sum([doc["num_chars"] for doc in corpus])
avg_chars = total_chars / len(corpus) if corpus else 0
longest_doc = max(corpus, key=lambda d: d["num_chars"])
shortest_doc = min(corpus, key=lambda d: d["num_chars"])

# Đếm số URL và Email tìm được trong raw_text
all_raw_text = " ".join([d["raw_text"] for d in corpus])
num_urls = len(re.findall(r'https?://\S+', all_raw_text))
num_emails = len(re.findall(r'[\w.-]+@[\w.-]+\.\w+', all_raw_text))

# Top 20 token thô xuất hiện nhiều nhất
all_tokens = all_raw_text.split()
top_20_tokens = Counter(all_tokens).most_common(20)

print("\n" + "=" * 60)
print("BÁO CÁO THỐNG KÊ CORPUS (CORPUS STATISTICS)")
print("=" * 60)
print("1. Số documents theo nguồn (Source):")
for src, count in sources_count.items():
    print(f"   - {src}: {count} documents")

print(f"2. Tổng số ký tự: {total_chars:,} ký tự")
print(f"3. Trung bình ký tự / document: {avg_chars:.1f} ký tự")
print(f"4. Document dài nhất: ID '{longest_doc['id']}' ({longest_doc['source']}) - {longest_doc['num_chars']} ký tự")
print(f"5. Document ngắn nhất: ID '{shortest_doc['id']}' ({shortest_doc['source']}) - {shortest_doc['num_chars']} ký tự")
print(f"6. Số URL tìm được trước cleaning: {num_urls}")
print(f"7. Số Email tìm được trước cleaning: {num_emails}")
print("8. Top 20 token thô xuất hiện nhiều nhất:")
for rank, (token, freq) in enumerate(top_20_tokens, 1):
    print(f"   {rank:2d}. '{token}': {freq}")

# ------------------------------------------------------------------------------
# YÊU CẦU E — QUALITY INSPECTION (Kiểm tra chất lượng trên 5 văn bản ngẫu nhiên)
# ------------------------------------------------------------------------------
print("\n" + "=" * 60)
print("KIỂM TRA CHẤT LƯỢNG (QUALITY INSPECTION - 5 DOCUMENTS NGẪU NHIÊN)")
print("=" * 60)

random.seed(42)  # Đảm bảo kết quả ổn định khi chạy lại
sample_docs = random.sample(corpus, min(5, len(corpus)))

for idx, doc in enumerate(sample_docs, 1):
    print(f"\n--- [Document Mẫu {idx}] ID: {doc['id']} | Nguồn: {doc['source']} | Tiêu đề: {doc['title']} ---")
    print("RAW TEXT (500 ký tự đầu):")
    print(doc['raw_text'][:500] + ("..." if len(doc['raw_text']) > 500 else ""))
    print("\nCLEAN TEXT (500 ký tự đầu):")
    print(doc['clean_text'][:500] + ("..." if len(doc['clean_text']) > 500 else ""))
    print("\nNhận xét thay đổi:")
    print("- Đã chuẩn hóa Unicode NFC, chuyển chữ thường (lowercase) và gom khoảng trắng thừa.")
    print("- Đã thay thế URL/Email (nếu có) thành token chuẩn <URL>, <EMAIL>.")
    print("- Trường hợp có thể mất thông tin: Việc lowercase làm mất dấu hiệu nhận diện tên riêng/viết tắt (NER) và dấu câu cảm xúc lặp lại.")

# ------------------------------------------------------------------------------
# YÊU CẦU F — OUTPUT (Lưu kết quả ra file clean_corpus.json)
# ------------------------------------------------------------------------------
output_file = Path("clean_corpus.json")
output_file.write_text(
    json.dumps(corpus, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print("\n" + "=" * 60)
print(f"[THÀNH CÔNG] Đã lưu {len(corpus)} documents vào file: {output_file.resolve()}")
print("=" * 60)


incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams
incorrect startxref pointer(1)
parsing for Object Streams


--> Tổng số documents thu thập được: 24

BÁO CÁO THỐNG KÊ CORPUS (CORPUS STATISTICS)
1. Số documents theo nguồn (Source):
   - TXT: 5 documents
   - JSON: 6 documents
   - PDF: 5 documents
   - DOCX: 5 documents
   - HTML/Web: 3 documents
2. Tổng số ký tự: 43,211 ký tự
3. Trung bình ký tự / document: 1800.5 ký tự
4. Document dài nhất: ID 'doc_023' (HTML/Web) - 21985 ký tự
5. Document ngắn nhất: ID 'doc_008' (JSON) - 108 ký tự
6. Số URL tìm được trước cleaning: 1
7. Số Email tìm được trước cleaning: 1
8. Top 20 token thô xuất hiện nhiều nhất:
    1. 'và': 169
    2. '[': 140
    3. ']': 140
    4. 'các': 139
    5. 'của': 120
    6. 'học': 108
    7. 'trong': 100
    8. 'là': 94
    9. 'có': 94
   10. 'được': 92
   11. 'một': 89
   12. 'hình': 85
   13. 'nhân': 79
   14. 'tạo': 79
   15. 'dữ': 79
   16. 'liệu': 78
   17. ',': 76
   18. 'thể': 72
   19. 'mô': 71
   20. 'máy': 58

KIỂM TRA CHẤT LƯỢNG (QUALITY INSPECTION - 5 DOCUMENTS NGẪU NHIÊN)

--- [Document Mẫu 1] ID: doc_021 | Nguồn: 

# 10. Câu hỏi thảo luận

Các câu hỏi sau **không có đáp án trong notebook**.

1. Unicode khác UTF-8 như thế nào?
2. Tại sao hai chuỗi nhìn giống nhau có thể không bằng nhau khi so sánh bằng Python?
3. Tại sao text extraction từ PDF thường khó hơn TXT?
4. PDF scan khác PDF có text layer như thế nào về mặt xử lý?
5. Tại sao không nên dùng `soup.get_text()` một cách máy móc để lấy nội dung bài báo?
6. Regex phù hợp và không phù hợp với những loại bài toán nào?
7. Có nên luôn chuyển text về lowercase không?
8. Có nên luôn xóa punctuation và emoji không?
9. Khi nào URL nên bị xóa, khi nào nên thay bằng `<URL>`?
10. Vì sao text cleaning phải phụ thuộc downstream NLP task?
11. Những lỗi nào trong preprocessing có thể ảnh hưởng tới mô hình NLP ở các bước sau?
12. Hãy đề xuất cách kiểm thử một text processing pipeline.

# 11. Checklist trước khi nộp

- [ ] Notebook chạy từ đầu đến cuối.
- [ ] Không hard-code kết quả.
- [ ] Có xử lý Unicode tiếng Việt.
- [ ] Có ít nhất 4 loại nguồn dữ liệu.
- [ ] Có regex.
- [ ] Có hàm cleaning.
- [ ] Có corpus statistics.
- [ ] Có kiểm tra chất lượng raw/clean text.
- [ ] Có `clean_corpus.json`.
- [ ] Có nhận xét về hạn chế của pipeline.

**Nộp:** Jupyter Notebook/Google Colab + dữ liệu cần thiết để chạy lại bài.